# Bazalt in a notebook

Bazalt needs no window and no display. It starts the window system only when you ask
for a `Window`, so a kernel on a remote server — a university machine with a GPU and
no screen — runs everything below.

This example is not run by CI, like every other example. Run it yourself.

It needs `pillow` to show the pictures and `ipywidgets` for the last cell:

    pip install bazalt pillow ipywidgets

**The one rule for a notebook:** use `with`, and read your pixels inside the block.
A cell you re-run otherwise leaves the previous Context's worker threads alive until
the garbage collector reaches them, and on a shared machine that adds up.

In [ ]:
import numpy as np
from PIL import Image

import bazalt as bz

# Which GPU the kernel can see. On a login node this is often the answer to
# "why is nothing working" — no device, no Vulkan driver installed.
for device in bz.list_devices():
    print(device)

## A picture in a cell

`target.color[0].read()` gives a NumPy array, and everything that shows a NumPy array
shows this one. Bazalt has no display verb of its own on purpose: PIL, matplotlib and
imageio already answer that question, and they let you pick the layer and the mip.

In [ ]:
VERTEX = """
#version 450
layout(location = 0) out vec2 uv;
void main() {
    // One triangle covering the screen. No vertex buffer.
    uv = vec2((gl_VertexIndex << 1) & 2, gl_VertexIndex & 2);
    gl_Position = vec4(uv * 2.0 - 1.0, 0.0, 1.0);
}
"""

FRAGMENT = """
#version 450
layout(location = 0) in vec2 uv;
layout(location = 0) out vec4 color;
layout(push_constant) uniform Push { float scale; } push;
void main() {
    vec2 p = (uv - 0.5) * push.scale;
    float r = length(p);
    float rings = 0.5 + 0.5 * sin(40.0 * r - 6.0 * atan(p.y, p.x));
    color = vec4(rings * vec3(0.4, 0.7, 1.0) + 0.1, 1.0);
}
"""


def render(scale=1.0, size=512):
    """Build a Context, draw one frame, give back the pixels.

    Everything lives inside the `with`, including the read. A closed Context
    starts no new work, so target.color[0].read() outside the block raises
    StateError instead of handing back a black image.
    """
    import struct

    with bz.Context() as ctx:
        pipeline = (ctx.graphics_pipeline()
                    .vertex_shader(ctx.compile_shader("ring.vert", bz.ShaderStage.VERTEX,
                                                      source=VERTEX))
                    .fragment_shader(ctx.compile_shader("ring.frag", bz.ShaderStage.FRAGMENT,
                                                        source=FRAGMENT))
                    .push_constant(4, bz.ShaderStage.FRAGMENT))
        target = ctx.create_render_target(size, size)
        built = pipeline.build(target)

        cmd = ctx.create_command_buffer()
        cmd.begin()
        with cmd.rendering(target, clear_color=[0.02, 0.02, 0.05, 1.0]) as c:
            c.bind_pipeline(built)
            c.push_constants(built, 0, struct.pack("f", scale))
            c.draw(3)
        ctx.submit(cmd)

        return target.color[0].read()


Image.fromarray(render())

## What a notebook is actually for

Not one picture — a sweep. A notebook is a parameter editor with a shader in it, and
this is where it beats a windowed loop: the shader source is a cell, so you edit
`FRAGMENT` above, re-run that cell, and drag the slider again.

This is plain Python. `ipywidgets` drives the same `render()` function; bazalt has no
notebook API and needs none.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

out = widgets.Output()
slider = widgets.FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1, description="scale")


def redraw(change):
    with out:
        out.clear_output(wait=True)
        display(Image.fromarray(render(scale=change["new"], size=384)))


slider.observe(redraw, names="value")
display(slider, out)
redraw({"new": slider.value})

## Compute, with no rendering at all

The other half a notebook is good for: the GPU as a calculator. A buffer in, a buffer
out, NumPy on both ends.

In [ ]:
COMPUTE = """
#version 450
layout(local_size_x = 64) in;
layout(set = 0, binding = 0) buffer Data { float values[]; };
void main() {
    uint i = gl_GlobalInvocationID.x;
    if (i < values.length()) {
        values[i] = sqrt(values[i]);
    }
}
"""

data = np.arange(1024, dtype=np.float32)

with bz.Context() as ctx:
    pipeline = (ctx.compute_pipeline()
                .shader(ctx.compile_shader("sqrt.comp", bz.ShaderStage.COMPUTE, source=COMPUTE))
                .storage_buffer(0)
                .build())
    buffer = ctx.create_buffer(data, bz.BufferType.STORAGE, bz.MemoryUsage.STATIC)
    pool = ctx.create_descriptor_pool()
    dset = pool.allocate_set(pipeline, set=0)
    dset.set_buffer(0, buffer)

    cmd = ctx.create_command_buffer()
    cmd.begin()
    cmd.bind_pipeline(pipeline).bind_descriptor_set(dset, pipeline, set=0).dispatch(1024 // 64)
    ctx.submit(cmd)

    result = buffer.read(np.float32)

print(result[:8])
print("matches numpy:", np.allclose(result, np.sqrt(data), atol=1e-5))

## What happens if you forget the `with`

The Context stays alive with its upload worker and its hot-reload watcher, until the
garbage collector reaches it. Run the cell ten times and you have ten of them.

`ctx.close()` is the same call without the block, for a Context you keep in a cell of
its own and want to release from another one. It is idempotent, and `ctx.closed` says
whether it has run.

In [ ]:
ctx = bz.Context()
target = ctx.create_render_target(32, 32)
print("device:", ctx.device_name, "| headless:", ctx.headless)

ctx.close()
print("closed:", ctx.closed)

# The cached facts still answer. Anything that would start new work does not.
print("device after close:", ctx.device_name)
try:
    target.color[0].read()
except bz.StateError as error:
    print("StateError:", error)